## Handling Long Conversations

We have used seen the concept of short-term memory, which helps your agent keep track of all the messages (Human, AI and Tool) that have been exchanged so far between Human & Agent. We enabled this simply by _attaching_ an instance of `InMemorySaver` class to the `create_agent()` call as shown below.

```python
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5-nano",
    ...
    checkpointer=InMemorySaver(),
)
```
With the help of this checkpointer, an Agents is able to keep track of all messages, so it can remember the complete thread of any conversation with the user. This work very well for _short_ conversations. After a while the list becomes longer & longer so that it overflows the size of our Agent's (actually the model of the Agent) context window - recall that each model has a pre-defined size of a context windows (for example GPT models is typically 400K tokens, Haiku is 200K and Sonnet/Open is 1 million and so on). At this point, the model _cannot literally_ parse all this information in an efficient way, thus slowing down our app, impacting performance and increasing its cost.

There are 2 ways we can solve this problem, both using middleware.
1. **Summarizing the conversation** so far, and
2. Trimming or deleting (older) messages

We will cover both these techniques in this workbook.

In [1]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

## Summarizing the Conversation

Summarizing the conversation can be done using _out-of-the-box_ middleware (viz. `langchain.agents.middleware.SummarizationMiddleware`). The `SummarizationMiddleware` summarizes conversation history when _set_ message limits or token limits are approached. This middleware monitors message or token counts, as appropriate, and automatically summarizes older messages when a threshold is reached, preserving recent messages and maintaining context continuity by ensuring AI/Tool message pairs remain together. 

Here is the definition of `SummarizationMiddleware` [@see [docs](https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware)]

```python
SummarizationMiddleware(
    model="openai:gpt-5-nano",
    trigger=...,
    keep=..., 
    token_counter="...", 
    summary_prompt=..., 
    trim_tokens_to_summarize = ...,
)
```

| Parameter | Meaning |
| :-- | :-- |
| `model` | Model used to summarize context - use string similar to the `model` parameter of `create_agent()` call. For example, `openai:gpt-5-nano` (need not be same as the model used by `create_agent()`) | 
| `trigger` | Event (or threshold) that triggers summarization of the context. <br/> Examples: <br/> `("messages", 50)` - fire when 50 messages threshold is reached in agent's checkpointer <br/> `("tokens", 3000)` - fire when 3000 tokens is reached <br/> `[("fraction", 0.8), ("messages", 100)]` - fire when either when 80% of model's max input tokens is reached or when 100 messages is reached (whichever comes first) |
| `keep` | How much of the context to keep. Defaults to keeping the most recent 20 _human_ messages. <br/> Examples: <br/> `("messages", 20)` - keep the most recent 20 _human_ messages (and related AI responses). So I get (<Summary of previous messages> + <20 Human messages + their AI responses>) as my summarized context <br/> `("tokens", 3000)` - keep the most recent 3000 tokens <br/> `("fraction", 0.3)` - Keep the most recent 30% of the model's max input tokens |
| `token_counter` | **(Optional)** - function to count the tokens in message. Defaults to `count_tokens_approximately`|
| `summary_prompt` | **(Optional)** - prompt template for generating summary. Defaults to `DEFAULT_SUMMARY_PROMPT` constant defined by LangChain|
| `trim_tokens_to_summarize` | **(Optional)** - Maximum tokens to keep when preparing messages for the summarization call. Defaults to `_DEFAULT_TRIM_TOKEN_LIMIT` defined internally by LangChain|

In most cases, we'll use the `model`, `triggers` and `keep` parameters leaving others at default values.

An example implementation is shown below. 

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware


agent = create_agent(
    model="openai:gpt-5-nano",
    # checkpointer is obviously required to keep track of message history!
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            # using a different (cheaper) model to summarize
            # it need not be different from main model, but model spec is required!
            model="openai:gpt-4o-mini",
            # fire when 100 tokens are reached
            trigger=("tokens", 100),
            # keep the most recent human message + AI response
            keep=("messages", 1),
        )
    ],
)

Let's simulate a large number of messages being passed in the context to our agent. This should (hopefully) trigger our Summarization middleware. We have _deliberately_ kept the thresholds small in the example above. In practice, these would be reasonably larger.

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

# simulate a conversation history
messages = [
    HumanMessage(
        content="I want to start building AI agents with LangChain. Where do I begin?"
    ),
    AIMessage(
        content="You should start with the concept of an 'AgentExecutor' and 'Tools'. An agent uses an LLM to decide which actions to take and in what order."
    ),
    HumanMessage(content="What exactly is a 'Tool' in this context?"),
    AIMessage(
        content="A tool is a specific function the agent can call, like a Google Search, a calculator, or a custom Python function you've written."
    ),
    HumanMessage(content="How does the LLM know which tool to pick?"),
    AIMessage(
        content="It relies on the tool's description. The LLM reads the descriptions of all available tools and chooses the one that best matches the user's intent."
    ),
    HumanMessage(content="Can you show me a simple code snippet for a custom tool?"),
    AIMessage(
        content='Sure! You use the @tool decorator: \n\n@tool\ndef get_word_length(word: str) -> int:\n    """Returns the length of a word."""\n    return len(word)'
    ),
    HumanMessage(
        content="Okay, so if I have tools, what is the 'Prompt Template' for an agent?"
    ),
    AIMessage(
        content="Agents usually use specific templates like 'hwchase17/react'. It includes sections for 'Thought', 'Action', 'Action Input', and 'Observation'."
    ),
    HumanMessage(content="What is 'ReAct' logic?"),
    AIMessage(
        content="ReAct stands for Reasoning and Acting. The agent thinks about what to do, takes an action, observes the result, and repeats until it has an answer."
    ),
    HumanMessage(
        content="I'm getting an error that the LLM isn't stopping. It just keeps hallucinating tool outputs."
    ),
    AIMessage(
        content="You likely need to ensure your 'Stop' sequences are set correctly so the LLM pauses after generating an 'Action Input'."
    ),
    HumanMessage(content="How do I add memory so the agent remembers previous steps?"),
    AIMessage(
        content="You can use 'ConversationBufferMemory' or 'SqliteMessageHistory' and pass it into the AgentExecutor using the 'memory' parameter."
    ),
    HumanMessage(
        content="Wait, I heard AgentExecutor is being deprecated. What's the alternative?"
    ),
    AIMessage(
        content="Great catch! LangChain is moving toward 'LangGraph'. It gives you much more control over the agent's state and flow compared to the old Executor."
    ),
    HumanMessage(content="Is LangGraph harder to learn?"),
    AIMessage(
        content="It has a steeper learning curve because you define the agent as a state machine (nodes and edges), but it's much more robust for production."
    ),
    HumanMessage(content="How do I handle a case where a tool returns an error?"),
    AIMessage(
        content="In LangGraph, you can create an edge that catches exceptions and routes them back to the LLM to 'self-correct' based on the error message."
    ),
    HumanMessage(content="Can I give an agent access to a local database?"),
    AIMessage(
        content="Yes, you can create a 'SQLDatabaseToolkit' which allows the agent to query schemas and execute SQL queries safely."
    ),
    HumanMessage(content="Is it safe to let an AI run SQL queries?"),
    AIMessage(
        content="It's risky. You should use a read-only user and always implement a 'human-in-the-loop' check for 'DROP' or 'DELETE' commands."
    ),
    HumanMessage(content="What is 'Human-in-the-loop' in LangChain?"),
    AIMessage(
        content="It's a breakpoint where the agent pauses and waits for a human to approve or edit the next action before proceeding."
    ),
    HumanMessage(
        content="How do I make the agent faster? It's taking forever to think."
    ),
    AIMessage(
        content="Try using a smaller model for simple routing tasks or use 'streaming' to show the agent's 'Thoughts' to the user in real-time."
    ),
    HumanMessage(
        content="What if the tool output is too long for the LLM's context window?"
    ),
    AIMessage(
        content="That's where your SummarizationMiddleware comes in! You can summarize tool outputs before feeding them back to the LLM."
    ),
    HumanMessage(content="Does the agent lose detail if I summarize the tool output?"),
    AIMessage(
        content="Potentially, yes. It's a trade-off. You should use a 'Map-Reduce' approach if you need to keep specific details from a large dataset."
    ),
    HumanMessage(content="Can an agent call another agent?"),
    AIMessage(
        content="Yes, this is called a 'Multi-Agent' system. You have a 'Supervisor' agent that delegates tasks to specialized 'Worker' agents."
    ),
    HumanMessage(content="How do multi-agent systems communicate?"),
    AIMessage(
        content="They share a 'State' object. One agent updates the state, and the next agent reads that state to perform its task."
    ),
    HumanMessage(
        content="What's the best way to debug an agent that's stuck in a loop?"
    ),
    AIMessage(
        content="Use 'LangSmith'. It provides a visual trace of every step, tool call, and LLM prompt so you can see exactly where it got confused."
    ),
    HumanMessage(content="Is LangSmith free?"),
    AIMessage(
        content="It has a generous free tier for personal projects, but it becomes paid for high-volume enterprise logging."
    ),
    HumanMessage(
        content="What's the difference between a 'Zero-shot' agent and a 'Structured' agent?"
    ),
    AIMessage(
        content="Zero-shot agents take one string input. Structured agents can handle tools that require multiple complex arguments."
    ),
    HumanMessage(content="Can I use local models like Llama 3 for these agents?"),
    AIMessage(
        content="Absolutely. You can use 'Ollama' or 'vLLM' to host the model and connect it to LangChain using the 'ChatOllama' class."
    ),
    HumanMessage(content="Does Llama 3 support tool calling as well as GPT-4?"),
    AIMessage(
        content="Llama 3 is quite good, but you need to use the specific 'Tool Calling' versions of the models for the best reliability."
    ),
    HumanMessage(content="Okay, I think I'm ready to build. Any final advice?"),
    AIMessage(
        content="Keep your tools atomic and your descriptions very clear. If a human couldn't understand the tool description, the AI won't either!"
    ),
]

config = {"configurable": {"thread_id": ""}}

response = agent.invoke(
    {"messages": messages},
    config=config,
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user wants to learn how to build AI agents using LangChain, including understanding key concepts, tools, and best practices for implementation.\n\n## SUMMARY\nThe user inquired about various aspects of building AI agents with LangChain, with key discussions including:\n- **AgentExecutor and Tools**: An agent uses an 'AgentExecutor' and tools to execute actions based on user intent.\n- **Tool Definition**: Tools are specific functions (e.g., Google Search, calculator) the agent can call and are selected based on their descriptions.\n- **Prompt Template**: Agents utilize templates like ReAct logic to guide their reasoning and actions.\n- **Avoiding Hallucinations**: Proper configuration of 'Stop' sequences can help mitigate issues where the LLM generates output continuously.\n- **Memory Management**: Options like 'ConversationBufferMemory' and 'SqliteMessageHistory' allow agents t

Notice that it has retained just the last HumanMessage + it's AI response. This is preceeded with a summary of the context.

Now if I print the first (at index 0) of the retained messages, I should see the summarized context. Notice that the model has done a fairly good job of summarizing the context - this provides enough context to the model to progress with the rest of the conversation.

In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user wants to learn how to build AI agents using LangChain, including understanding key concepts, tools, and best practices for implementation.

## SUMMARY
The user inquired about various aspects of building AI agents with LangChain, with key discussions including:
- **AgentExecutor and Tools**: An agent uses an 'AgentExecutor' and tools to execute actions based on user intent.
- **Tool Definition**: Tools are specific functions (e.g., Google Search, calculator) the agent can call and are selected based on their descriptions.
- **Prompt Template**: Agents utilize templates like ReAct logic to guide their reasoning and actions.
- **Avoiding Hallucinations**: Proper configuration of 'Stop' sequences can help mitigate issues where the LLM generates output continuously.
- **Memory Management**: Options like 'ConversationBufferMemory' and 'SqliteMessageHistory' allow agents to remember previous interactions.
- **Transitio

Using this one `SummarizationMiddleware` middleware call, we were able to summarize all the messages in the context, depending on the thresholds we provide.

## Adding Custom Middleware

LangChain supports adding _custom middleware_, which is implemented using Python functions that are decorated with `@before_XXXX` or `@after_XXXX` decorators to signal that they are called before or after a specific stop in the Agent workflow. For example: `@before_agent` means fire before agent is called. The diagram shows below shows you all the _intervention_ points where you can add custom middleware.

| Core Agent Loop | Where You can _insert_ middleware 'hooks' |
| :-- | :-- |
| The core agent loop wrapped by `create_agent()` call<br/> <div align="center"> <img src="images/01_core_agent_loop.avif" width="250" heigh="100" alt="Core Agent Loop"/> </div> | The middleware _hooks_ that we can apply _before_ and/or _after_ each step <br/> <div align="center"> <img src="images/08_middleware_final.avif" width="250" heigh="100" alt="Middleware"/> </div> |

Following are the decorators we can apply:
| Annotation | When called | Practical Use-case |
| :-- | :-- | :-- |
| `@before_agent` or `@abefore_agent` | Logic to run _before_ the agent execution starts. The `@abefore_agent` annotates an async function. | Injecting a user's unique "Persona" or system constraints into the message history before the agent even starts thinking. |
| `@before_model` or `@abefore_model` | Logic to run _before_ model is calls. The `@abefore_model` annotates an async function. | Redacting PII (Personally Identifiable Information) or checking if the prompt violates safety guidelines before sending it to the LLM (to save costs/latency). |
| `@after_model` or `@aafter_model` | Logic to run _after_ model is calls completes. The `@aafter_model` annotates an async function. | Parsing a JSON response from the model to ensure it follows a specific schema. If it fails, you can raise an error or flag it for a retry. |
| `@wrap_model_call` or `@awrap_model_call` | intercept and control model execution via handler callback. The `@awrap_model_call` annotates an async function | Checking a Redis cache to see if this exact prompt was asked recently. If yes, return the cached result; if not, proceed with the model call. |
| `@after_agent` or `@aafter_agent` | Logic to run _agent_ the agent execution completes. The `@abefore_agent` annotates an async function. | Logging the final result to an external database (like LangSmith) or generating a "Summary" of the entire interaction for a UI. |
| `@wrap_tool_call` or `@awrap_tool_call` | intercept tool execution for retries, monitoring or modification. The `@awrap_tool_call` annotates an async function. | If a "Search" tool fails due to a rate limit (429 error), this logic can implement an exponential backoff before the agent sees the failure. |

Now let's see code examples of each type of annotation below:

### @before_agent example

The `@before_agent` hook fires **once**, right before the agent's processing loop begins (i.e., before the first model call). The user's `HumanMessage` is already in state at that point, giving you the opportunity to **inject a personalised `SystemMessage`** built from data in an auth system or user-profile database — _before the agent starts thinking_.

The decorated function must accept `(state: AgentState, runtime: Runtime)` and return either a `dict` of state updates or `None`.

> **Note:** This hook fires **only once per `invoke()` call**, not on every turn of the model-tool loop. For logic that must run before every model call, use `@before_model` instead.


Below we illustrate a **persona injection example**: instead of hard-coding a system prompt inside `create_agent()` call, you look up the current user's profile from an auth system or CRM _at runtime_ and _build a personalised SystemMessage_ on the fly. In the example below, the middleware fetches a profile for `user_001` (name, subscription tier, product, open tickets) and injects it as a SystemMessage before the agent starts thinking — so the model greets the customer by name, is aware of their plan tier, and knows to offer Premium escalation paths without any of that logic living in the agent definition itself.

In [2]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)

True

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_agent, AgentState
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from rich.console import Console

console = Console()

In [8]:
# --------------------------------------------------------------------------
# Simulated user-profile database (normally queried from an auth/CRM system)
# --------------------------------------------------------------------------
USER_PROFILES = {
    "user_001": {
        "name": "Priya Sharma",
        "tier": "Premium",
        "product": "CloudDB Pro",
        "open_tickets": 2,
    },
    "user_002": {
        "name": "Tom Zhang",
        "tier": "Basic",
        "product": "CloudDB Starter",
        "open_tickets": 0,
    },
}

In [9]:
# --------------------------------------------------------------------------
# @before_agent middleware — fires once before the agent loop starts
# --------------------------------------------------------------------------
@before_agent
def inject_user_persona(state: AgentState, runtime: Runtime) -> dict | None:
    # In production, user_id comes from the request session / auth token
    user_id = "user_001"
    profile = USER_PROFILES[user_id]

    print(f"  [@before_agent] Hook CALLED!")
    print(f"  [@before_agent] State currently has {len(state['messages'])} message(s).")
    print(f"  [@before_agent] Looking up profile for user_id='{user_id}'...")
    print(
        f"  [@before_agent] Found : {profile['name']} | Tier: {profile['tier']} | Product: {profile['product']}"
    )

    persona = SystemMessage(
        content=(
            f"You are a helpful support agent for CloudDB. "
            f"The customer is {profile['name']} on the {profile['tier']} plan, "
            f"using {profile['product']}. "
            f"They currently have {profile['open_tickets']} open support ticket(s). "
            f"Always address the customer by their first name. "
            f"Premium customers should be offered priority escalation paths when relevant."
        )
    )

    print(f"  [@before_agent] Injecting SystemMessage (persona) into state:")
    print(f'  [@before_agent]   -> "{persona.content}"')
    print(
        f"  [@before_agent] State will grow from {len(state['messages'])} → {len(state['messages']) + 1} message(s).\n"
    )
    return {"messages": [persona]}

In [10]:
# --------------------------------------------------------------------------
# Create agent with the @before_agent middleware attached
# --------------------------------------------------------------------------
agent = create_agent(
    model="openai:gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[inject_user_persona],
)

In [11]:
# invoke the agent...
config = {"configurable": {"thread_id": "before-agent-demo"}}

print("=" * 65)
print("Invoking agent — watch @before_agent fire before the model call")
print("=" * 65 + "\n")

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "I can't connect to my database from my application. "
                "What should I check first?"
            )
        ]
    },
    config=config,
)

print("-" * 65)
print("Agent Response:")
print("-" * 65)
console.print(response["messages"][-1].content)

Invoking agent — watch @before_agent fire before the model call

  [@before_agent] Hook CALLED!
  [@before_agent] State currently has 1 message(s).
  [@before_agent] Looking up profile for user_id='user_001'...
  [@before_agent] Found : Priya Sharma | Tier: Premium | Product: CloudDB Pro
  [@before_agent] Injecting SystemMessage (persona) into state:
  [@before_agent]   -> "You are a helpful support agent for CloudDB. The customer is Priya Sharma on the Premium plan, using CloudDB Pro. They currently have 2 open support ticket(s). Always address the customer by their first name. Premium customers should be offered priority escalation paths when relevant."
  [@before_agent] State will grow from 1 → 2 message(s).

-----------------------------------------------------------------
Agent Response:
-----------------------------------------------------------------


Hi Priya,

I’m sorry you’re hitting a connection issue. Here are the first things I’d check to triage quickly. You can run 
through these in order and share any error messages you see.

1) Verify the database is up and you’re hitting the right endpoint
- In CloudDB Console, confirm the instance shows as Running and note the endpoint host and port.
- If you’re using a private endpoint/VPC, ensure you’re using the private address and that the endpoint is 
reachable from your app network.

2) Check credentials and user permissions
- Confirm the username and password (and any required database name) are correct.
- If you’ve rotated credentials recently, update them in your application.
- Ensure the user has the necessary privileges on the target database.

3) Confirm network access and firewall rules
- If you’re connecting from a public IP, ensure that IP is allowed in CloudDB security groups/firewalls.
- If you’re in a VPC or using private endpoints, verify VPC peering/VPN routing and that DNS resolves to the 
private endpoint when required.
- Check for any recently changed network policies or corporate firewall rules.

4) Validate the connection string and TLS settings
- Double-check the connection string format for your DB type (PostgreSQL, MySQL, etc.). Make sure host, port, 
database name, user, and flags (sslmode, requireSSL, etc.) are correct.
- If SSL/TLS is required, ensure you’re using the proper SSL mode and CA certificate (and that your client supports
the required TLS version).

5) Test connectivity from the app host (or a nearby host)
- From the same machine/container where the app runs, try a direct connect with a simple client:
  - PostgreSQL: psql "host=HOST port=PORT dbname=DB user=USER password=PASSWORD sslmode=require"
  - MySQL: mysql -h HOST -P PORT -u USER -pDBPASSWORD DB
- If direct connection fails, note the exact error (e.g., could not connect, authentication failed, timeout, 
network unreachable).

6) Check for common errors and logs
- Connection timeout vs. authentication error vs. network error: note the exact message and error code.
- Look at your application logs for stack traces and any DB driver message.
- If available, check CloudDB metrics/logs for connection limits, row locks, or maintenance events.

7) DNS and routing sanity
- Resolve the endpoint from your app host (nslookup/dig) to ensure it's resolving to the expected address.
- If you’re using private DNS or split-horizon DNS, confirm the correct resolution in your environment.

8) Maintenance or service status
- Check CloudDB status or any recent maintenance notices in the region of your instance.

If you can share:
- The database type (PostgreSQL, MySQL, etc.)
- The CloudDB region and endpoint host/port (redacted if needed)
- The exact error message and any error codes
- Whether you’re connecting from inside a VPC/private network or over the internet
- Any recent changes (config, credentials, network, deployment)

I can give you targeted guidance based on that.

Premium escalation option
As a Premium/CloudDB Pro customer with two open tickets, you have priority escalation paths. If you’d like, I can 
route this to Priority Support right away. Just say “Yes, escalate to Priority Support,” and share the error 
details above and the DB type. I’ll initiate the fastest possible channel and keep you updated.

Would you like me to start a Priority escalation now, or would you prefer we work through the questions above 
first?

I am trying to build practical examples for EACH of the @before_XXX and @after_XXX and @wrap_XXX annotations shown in the table under the "Delete Older Messages" section. Don't go by the heading - examples NEED NOT BE related to "Deleting older messages" at all. I'd rather you give me more realistic/practical examples on how each of the annotation can be used. It would be best if you can generate COMPLETE (i.e. create_agent(), invoke() call and display response) for each of the example shown in the "Practical Use-case" column for the respective annotation. You can choose to use either the async or non-async call. Also, to keep things shorter, you can use one common `create_agent()` call across all these example, if that makes sense, but use separate invoke() calls with appropriate input messages & display the response - it should clearly show the annotation in action (maybe add some print function in the annotated function to show that it is called!).

Can you generate separate examples for each type of annotation? I want to add these examples under the decorators table to complete this notebook. Examples should use the version of LangChain I have used here.


Looks like it is taking too long. Let's go step by step - for now just add an example of using either @before_agent or @abefore_agent annotation to show how we can inject a user's unique persona or system constraints into message history before agents starts thinking. Example should be able to show (maybe by using print functions inside the annotated function) that the annotated function was actually called and what changes it made, so user gets a complete understanding. We'll add examples of other annotations later.

### @before_model example

The `@before_model` hook fires **before every call to the LLM** — on every turn of the model-tool loop (unlike `@before_agent`, which fires only once per `invoke()` call).

Like all hooks, the decorated function accepts **`(state: AgentState, runtime: Runtime)`** and returns `dict | None` (state updates). To read the current messages, access `state["messages"]`. To replace a message with a redacted version, return `{"messages": [new_msg]}` where `new_msg` carries the **same `id`** as the original — LangGraph's `add_messages` reducer updates by ID, so the original is replaced in-place rather than appended.

> **Note:** Because this hook fires on *every* model call, it will also see turns generated during tool use, not just the first human message. The example below therefore inspects only `HumanMessage` instances.

Below we illustrate a **PII-redaction + safety-guardrail example**. The middleware does two things before the message list is forwarded to the LLM:

1. **Safety guardrail** — scans the message for blocked topics (e.g. _"how to hack"_). If a violation is found, a `ValueError` is raised immediately, stopping the request _before_ an API call is made (saving cost and latency).
2. **PII redaction** — scans every `HumanMessage` for emails, phone numbers, SSNs, and credit-card numbers using regex and replaces each match with a safe token (`[EMAIL REDACTED]` etc.). The raw PII never leaves your infrastructure.

We run the agent twice: once with a message that contains PII, and once with a message that violates the safety policy.

In [15]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)

True

In [16]:
import re
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, AgentState
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from rich.console import Console

console = Console()

In [17]:
# --------------------------------------------------------------------------
# PII regex patterns — (compiled_pattern, replacement_token) tuples
# --------------------------------------------------------------------------
PII_PATTERNS = {
    "EMAIL": (
        re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
        "[EMAIL REDACTED]",
    ),
    "PHONE": (
        re.compile(r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"),
        "[PHONE REDACTED]",
    ),
    "SSN": (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[SSN REDACTED]"),
    "CREDIT CARD": (re.compile(r"\b(?:\d{4}[-\s]?){3}\d{4}\b"), "[CARD REDACTED]"),
}

# --------------------------------------------------------------------------
# Safety policy — topics whose presence blocks the request entirely
# --------------------------------------------------------------------------
BLOCKED_TOPICS = [
    "how to hack",
    "build a bomb",
    "make malware",
    "create ransomware",
    "synthesise drugs",
]


# --------------------------------------------------------------------------
# @before_model middleware — fires before EVERY model call
# Signature: same as @before_agent — (state: AgentState, runtime: Runtime)
# To replace a message, return a new HumanMessage with the SAME id — LangGraph's
# add_messages reducer updates by ID, replacing the original in-place.
# --------------------------------------------------------------------------
@before_model
def pii_and_safety_guard(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    print(f"\n  [@before_model] Hook CALLED!")
    print(
        f"  [@before_model] Intercepting {len(messages)} message(s) before they reach the LLM."
    )

    updates: list[BaseMessage] = []
    redacted_items: list[tuple[str, str]] = []

    for msg in messages:
        if isinstance(msg, HumanMessage):
            content = msg.content

            # --- 1. Safety check (fast: done before any copy is made) ---
            lower = content.lower()
            for topic in BLOCKED_TOPICS:
                if topic in lower:
                    print(
                        f"  [@before_model] SAFETY VIOLATION DETECTED: prompt contains '{topic}'"
                    )
                    print(
                        f"  [@before_model] Blocking request — LLM will NOT be called.\n"
                    )
                    raise ValueError(
                        f"Safety policy violation: blocked topic '{topic}' found in user message."
                    )

            # --- 2. PII redaction ---
            original_content = content
            for pii_type, (pattern, token) in PII_PATTERNS.items():
                matches = pattern.findall(content)
                if matches:
                    for m in matches:
                        redacted_items.append((pii_type, m))
                    content = pattern.sub(token, content)

            if content != original_content:
                # Same id → LangGraph replaces the original in-place (not appended)
                updates.append(HumanMessage(content=content, id=msg.id))

    if redacted_items:
        print(
            f"  [@before_model] PII DETECTED — {len(redacted_items)} item(s) redacted:"
        )
        for pii_type, original in redacted_items:
            print(f"    [{pii_type}] '{original}'  ->  replaced with token")
        print(
            f"  [@before_model] Forwarding REDACTED message to LLM. Original values never leave this node.\n"
        )
        return {"messages": updates}
    else:
        print(
            f"  [@before_model] No PII found. Safety check passed. Forwarding message unchanged.\n"
        )
        return None

In [18]:
# --------------------------------------------------------------------------
# Create agent — middleware handles PII + safety before every model call
# --------------------------------------------------------------------------
agent_pii_guard = create_agent(
    model="openai:gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[pii_and_safety_guard],
)

In [19]:
# --------------------------------------------------------------------------
# Invoke 1 — message contains PII: middleware should redact before LLM call
# --------------------------------------------------------------------------
user_msg_pii = (
    "Hi, I'm locked out of my account. My email is alice.morgan@gmail.com, "
    "my phone is 415-867-5309, and my SSN is 987-65-4321. "
    "Can you help me verify my identity and reset my password?"
)

print("=" * 68)
print("Invoke 1 — user message contains PII")
print("=" * 68)
print(f"\nOriginal user message (what the user typed):")
print(f'  "{user_msg_pii}"\n')

config1 = {"configurable": {"thread_id": "before-model-pii-demo"}}

response1 = agent_pii_guard.invoke(
    {"messages": [HumanMessage(user_msg_pii)]},
    config=config1,
)

print("-" * 68)
print("Agent Response (LLM saw the REDACTED version — PII never sent to API):")
print("-" * 68)
console.print(response1["messages"][-1].content)

Invoke 1 — user message contains PII

Original user message (what the user typed):
  "Hi, I'm locked out of my account. My email is alice.morgan@gmail.com, my phone is 415-867-5309, and my SSN is 987-65-4321. Can you help me verify my identity and reset my password?"


  [@before_model] Hook CALLED!
  [@before_model] Intercepting 1 message(s) before they reach the LLM.
  [@before_model] PII DETECTED — 3 item(s) redacted:
    [EMAIL] 'alice.morgan@gmail.com'  ->  replaced with token
    [PHONE] '415-867-5309'  ->  replaced with token
    [SSN] '987-65-4321'  ->  replaced with token
  [@before_model] Forwarding REDACTED message to LLM. Original values never leave this node.

--------------------------------------------------------------------
Agent Response (LLM saw the REDACTED version — PII never sent to API):
--------------------------------------------------------------------


I’m glad to help, but I can’t verify your identity or reset passwords from here. Also, please don’t share highly 
sensitive info (like your SSN) in chat. The safest path is to use the service’s official account recovery process. 
I can guide you step by step and help you draft a message to support if you’d like.

What you can do now
- Go to the service’s login page and look for links like “Forgot password,” “Can’t access your account?” or 
“Account recovery.”
- Start with the password reset flow. Enter your registered email or username. If you can access recovery options 
(backup email, phone number, authenticator app), use those to verify and reset.
- If you’re locked out of those options, look for the official “Account recovery” form or support contact. This is 
where you’ll verify your identity through official channels.

What information you’ll likely need (have ready)
- The account email or username on file
- Any recovery methods you still have (backup email, phone, authenticator app)
- Rough details about the account: approximate date of creation, last successful login, devices you’ve used to sign
in
- Any recent payments, subscriptions, or services tied to the account (do not share sensitive data in chat; use 
only what the service asks for)
- Government ID or other documents only if the service explicitly requests them through their secure channel

What to do after you regain access
- Change your password to a strong, unique one (not used anywhere else)
- Enable two-factor authentication (2FA) if available
- Review recent activity and sign out of all other sessions
- Update your recovery options (alternate email, phone, backup codes)

Optional: ready-to-send support request
If you’d like, I can draft a concise message you can send to the service’s support team. Tell me the service name 
(e.g., “Acme Mail,” “XYZ Social,” etc.) and I’ll tailor the message. Here’s a generic template you can start with:

Subject: Urgent account recovery and password reset request

Hello,
I’m locked out of my account and cannot complete the password reset through the standard login flow. My registered 
email is , and my phone number on file is , but I no longer have access to them. I’m willing to provide any 
information needed to verify my identity through your official channels. Please guide me through the identity 
verification process and help me reset my password. I will not share sensitive data in unsecured channels and will 
follow your secure verification steps.

Thank you,
[Your name]

Would you like me to tailor instructions to a specific service? If you tell me which service you’re trying to 
recover (and the platform you’re using: web, iOS, Android, etc.), I’ll give you exact steps and a ready-to-send 
message.

In [20]:
# --------------------------------------------------------------------------
# Invoke 2 — message violates safety policy: middleware should block it
# --------------------------------------------------------------------------
unsafe_msg = "Can you walk me through how to hack into a corporate server?"

print("=" * 68)
print("Invoke 2 — user message contains a safety violation")
print("=" * 68)
print(f"\nUser message:")
print(f'  "{unsafe_msg}"\n')

config2 = {"configurable": {"thread_id": "before-model-safety-demo"}}

try:
    response2 = agent_pii_guard.invoke(
        {"messages": [HumanMessage(unsafe_msg)]},
        config=config2,
    )
    console.print(response2["messages"][-1].content)
except ValueError as err:
    print("-" * 68)
    print("Request BLOCKED by @before_model middleware — no LLM call was made.")
    print("-" * 68)
    print(f"  Reason: {err}")

Invoke 2 — user message contains a safety violation

User message:
  "Can you walk me through how to hack into a corporate server?"


  [@before_model] Hook CALLED!
  [@before_model] Intercepting 1 message(s) before they reach the LLM.
  [@before_model] SAFETY VIOLATION DETECTED: prompt contains 'how to hack'
  [@before_model] Blocking request — LLM will NOT be called.

--------------------------------------------------------------------
Request BLOCKED by @before_model middleware — no LLM call was made.
--------------------------------------------------------------------
  Reason: Safety policy violation: blocked topic 'how to hack' found in user message.


### @wrap_model_call example

The `@wrap_model_call` hook **wraps every model invocation**, giving you a `handler` callable that represents the actual model call. You can:
- **short-circuit** by returning *without* calling `handler` (e.g. cache hit — no API call made)
- **call `handler(request)`** to execute the real model (cache miss)
- **call `handler` multiple times** for retry or fallback logic

The decorated function accepts **(request: ModelRequest, handler: Callable)** and must return either a `ModelResponse` or an `AIMessage` (which is auto-converted). The `ModelRequest` object carries:

| Attribute | Contents |
| :-- | :-- |
| `request.messages` | List of messages being sent to the LLM |
| `request.model` | The `BaseChatModel` instance |
| `request.state` / `request.runtime` | Agent state and runtime context |
| `request.override(**kwargs)` | Creates a modified copy (use for model fallbacks) |

Below we build a **Redis prompt cache**: the middleware hashes the last `HumanMessage` to form a deterministic cache key, checks Redis, and short-circuits on a hit — saving the latency and cost of a real API call. On a cache miss it calls the model, stores the response with a TTL, and returns normally.

We invoke the agent **three times**: first call is a **cache miss** (LLM called and result stored); second call is the **identical question** (cache hit — no LLM call); third call is a **new question** (cache miss again).

#### Installation

This example uses **`fakeredis`** — a pure-Python, in-memory Redis emulator that exposes the full `redis-py` API and requires **no Redis server**. It is ideal for local development and learning.

Install both packages into your `uv`-managed environment:

```bash
uv add redis fakeredis
```

**To switch to a real Redis server later** (one-command setup via Docker):

```bash
docker run -d -p 6379:6379 --name redis-dev redis:alpine
```

Then replace the two `fakeredis` lines in the setup cell below with:

```python
import redis
redis_client = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
```

Everything else stays the same — `fakeredis` and `redis-py` share the same API.

In [21]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [22]:
import hashlib
import json
from datetime import datetime, timezone
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich.console import Console
import fakeredis

console = Console()

In [23]:
# --------------------------------------------------------------------------
# Redis client setup
# Using fakeredis for local demo — no Redis server installation required.
# To use a real Redis server, replace these two lines with:
#   import redis
#   redis_client = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
# --------------------------------------------------------------------------
redis_client = fakeredis.FakeRedis(decode_responses=True)

CACHE_TTL_SECONDS = 300  # 5 minutes

print(f"Redis client ready : {type(redis_client).__name__}")
print(f"Cache TTL          : {CACHE_TTL_SECONDS}s ({CACHE_TTL_SECONDS // 60} min)")

Redis client ready : FakeRedis
Cache TTL          : 300s (5 min)


In [24]:
def _cache_key(request: ModelRequest) -> str:
    """Stable SHA-256 cache key derived from the last HumanMessage in the request."""
    last_human = next(
        (
            m.content.strip().lower()
            for m in reversed(request.messages)
            if isinstance(m, HumanMessage)
        ),
        "",
    )
    digest = hashlib.sha256(last_human.encode()).hexdigest()[:16]
    return f"llm_cache:{digest}"


# --------------------------------------------------------------------------
# @wrap_model_call middleware — wraps every model invocation
# --------------------------------------------------------------------------
@wrap_model_call
def redis_cache_middleware(request: ModelRequest, handler) -> AIMessage:
    key = _cache_key(request)
    last_human = next(
        (m.content for m in reversed(request.messages) if isinstance(m, HumanMessage)),
        "<no human message>",
    )

    print(f"\n  [@wrap_model_call] Hook CALLED!")
    print(f"  [@wrap_model_call] Last human message : '{last_human[:65]}...'")
    print(f"  [@wrap_model_call] Cache key          : {key}")

    # --- Check Redis for a cached response ---
    cached = redis_client.get(key)
    if cached:
        data = json.loads(cached)
        ttl = redis_client.ttl(key)
        print(
            f"  [@wrap_model_call] CACHE HIT  — cached at {data['cached_at']}  (TTL remaining: {ttl}s)"
        )
        print(
            f"  [@wrap_model_call] Returning cached result. LLM API was NOT called.\n"
        )
        return AIMessage(
            content=f"[From cache — cached at {data['cached_at']}]\n\n{data['content']}"
        )

    # --- Cache miss: call the real model, then store the result ---
    print(f"  [@wrap_model_call] CACHE MISS — forwarding request to LLM...")
    response = handler(request)
    ai_msg = response.result[0]

    payload = json.dumps(
        {
            "content": ai_msg.content,
            "cached_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
            "model": type(request.model).__name__,
        }
    )
    redis_client.setex(key, CACHE_TTL_SECONDS, payload)
    print(
        f"  [@wrap_model_call] LLM responded. Result stored in Redis with TTL={CACHE_TTL_SECONDS}s.\n"
    )

    return ai_msg

In [25]:
# --------------------------------------------------------------------------
# Create agent with the Redis cache middleware attached
# --------------------------------------------------------------------------
agent_cached = create_agent(
    model="openai:gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[redis_cache_middleware],
)

In [26]:
# --------------------------------------------------------------------------
# Invoke 1 — first time asking this question: expect a CACHE MISS
# --------------------------------------------------------------------------
question = "What are the top 3 best practices for writing clean Python code?"

print("=" * 70)
print("Invoke 1 — first ask: CACHE MISS expected, LLM will be called")
print("=" * 70)
print(f"\nQuestion: '{question}'\n")

config1 = {"configurable": {"thread_id": "redis-cache-demo-1"}}

response1 = agent_cached.invoke(
    {"messages": [HumanMessage(question)]},
    config=config1,
)

print("-" * 70)
print("Response (freshly generated by LLM):")
print("-" * 70)
console.print(response1["messages"][-1].content)

Invoke 1 — first ask: CACHE MISS expected, LLM will be called

Question: 'What are the top 3 best practices for writing clean Python code?'


  [@wrap_model_call] Hook CALLED!
  [@wrap_model_call] Last human message : 'What are the top 3 best practices for writing clean Python code?...'
  [@wrap_model_call] Cache key          : llm_cache:ecd7dabfe2a18354
  [@wrap_model_call] CACHE MISS — forwarding request to LLM...
  [@wrap_model_call] LLM responded. Result stored in Redis with TTL=300s.

----------------------------------------------------------------------
Response (freshly generated by LLM):
----------------------------------------------------------------------


Here are three high-impact best practices for clean Python code:

1) Prioritize readability and consistency (PEP 8)
- Use meaningful names (functions, variables, classes).
- Keep functions small and focused (single responsibility).
- Follow standard formatting: line length (often 79–88), imports at top, consistent spacing, and clear docstrings 
for public APIs.
- Use tooling: black for formatting, isort for imports, flake8/ruff for linting. Consider a pre-commit hook.

2) Build modular, maintainable structure
- Decompose problems into small, well-defined functions and classes with clear interfaces.
- Minimize global state and side effects; favor pure or well-contained functions.
- Avoid duplication; refactor common patterns into reusable utilities or modules.
- Use explicit, stable APIs and keep coupling low between components.

3) Embrace testing and typing
- Write tests that exercise behavior and edge cases (unit tests, and some integration tests).
- Use type hints (PEP 484) to catch type errors early; run a static type checker like mypy.
- Document behavior via docstrings for public functions/classes.
- Set up CI to run tests and type checks automatically.

If you want, I can tailor these to a specific project (web app, data science, CLI tool) and suggest a minimal 
setup.

In [27]:
# --------------------------------------------------------------------------
# Invoke 2 — IDENTICAL question: expect a CACHE HIT, LLM NOT called
# --------------------------------------------------------------------------
print("=" * 70)
print("Invoke 2 — same question again: CACHE HIT expected, no LLM call")
print("=" * 70)
print(f"\nQuestion: '{question}'  (same as Invoke 1)\n")

config2 = {"configurable": {"thread_id": "redis-cache-demo-2"}}

response2 = agent_cached.invoke(
    {"messages": [HumanMessage(question)]},
    config=config2,
)

print("-" * 70)
print("Response (notice the '[From cache...]' prefix — LLM was NOT called):")
print("-" * 70)
console.print(response2["messages"][-1].content)

Invoke 2 — same question again: CACHE HIT expected, no LLM call

Question: 'What are the top 3 best practices for writing clean Python code?'  (same as Invoke 1)


  [@wrap_model_call] Hook CALLED!
  [@wrap_model_call] Last human message : 'What are the top 3 best practices for writing clean Python code?...'
  [@wrap_model_call] Cache key          : llm_cache:ecd7dabfe2a18354
  [@wrap_model_call] CACHE HIT  — cached at 2026-04-23 11:53:49 UTC  (TTL remaining: 276s)
  [@wrap_model_call] Returning cached result. LLM API was NOT called.

----------------------------------------------------------------------
Response (notice the '[From cache...]' prefix — LLM was NOT called):
----------------------------------------------------------------------


[From cache — cached at 2026-04-23 11:53:49 UTC]

Here are three high-impact best practices for clean Python code:

1) Prioritize readability and consistency (PEP 8)
- Use meaningful names (functions, variables, classes).
- Keep functions small and focused (single responsibility).
- Follow standard formatting: line length (often 79–88), imports at top, consistent spacing, and clear docstrings 
for public APIs.
- Use tooling: black for formatting, isort for imports, flake8/ruff for linting. Consider a pre-commit hook.

2) Build modular, maintainable structure
- Decompose problems into small, well-defined functions and classes with clear interfaces.
- Minimize global state and side effects; favor pure or well-contained functions.
- Avoid duplication; refactor common patterns into reusable utilities or modules.
- Use explicit, stable APIs and keep coupling low between components.

3) Embrace testing and typing
- Write tests that exercise behavior and edge cases (unit tests, and some integration tests).
- Use type hints (PEP 484) to catch type errors early; run a static type checker like mypy.
- Document behavior via docstrings for public functions/classes.
- Set up CI to run tests and type checks automatically.

If you want, I can tailor these to a specific project (web app, data science, CLI tool) and suggest a minimal 
setup.

In [28]:
# --------------------------------------------------------------------------
# Invoke 3 — a DIFFERENT question: expect a CACHE MISS again
# --------------------------------------------------------------------------
different_question = "In one sentence, what is the purpose of Python's GIL?"

print("=" * 70)
print("Invoke 3 — different question: CACHE MISS expected, LLM will be called")
print("=" * 70)
print(f"\nQuestion: '{different_question}'\n")

config3 = {"configurable": {"thread_id": "redis-cache-demo-3"}}

response3 = agent_cached.invoke(
    {"messages": [HumanMessage(different_question)]},
    config=config3,
)

print("-" * 70)
print("Response (freshly generated by LLM — new key, no cache entry yet):")
print("-" * 70)
console.print(response3["messages"][-1].content)

print("\n" + "=" * 70)
print(f"Redis cache now holds {redis_client.dbsize()} key(s):")
for k in redis_client.keys("llm_cache:*"):
    print(f"  {k}  (TTL: {redis_client.ttl(k)}s)")

Invoke 3 — different question: CACHE MISS expected, LLM will be called

Question: 'In one sentence, what is the purpose of Python's GIL?'


  [@wrap_model_call] Hook CALLED!
  [@wrap_model_call] Last human message : 'In one sentence, what is the purpose of Python's GIL?...'
  [@wrap_model_call] Cache key          : llm_cache:e80de78b183633f3
  [@wrap_model_call] CACHE MISS — forwarding request to LLM...
  [@wrap_model_call] LLM responded. Result stored in Redis with TTL=300s.

----------------------------------------------------------------------
Response (freshly generated by LLM — new key, no cache entry yet):
----------------------------------------------------------------------


To ensure that only one thread executes Python bytecode at a time, protecting access to Python objects and 
simplifying CPython's memory management.


Redis cache now holds 2 key(s):
  llm_cache:ecd7dabfe2a18354  (TTL: 257s)
  llm_cache:e80de78b183633f3  (TTL: 300s)
